## 1. Project Introduction

In [1]:
"""
AI Career & Resume Assistant — Phase 1 Backend
Modules: Profile, Resume Studio, JD Analyzer, Resume–JD Match, Skill Gap,
Career Mentor, Learning Roadmap, Interview Studio, Application Studio,
PDF/Document Studio.
LLM: Google Gemini (free tier) via google-generativeai.
"""
print("Phase 1 backend loaded.")

Phase 1 backend loaded.


## 2. Installation & Dependencies

In [2]:
!pip install -q -U google-genai pypdf fpdf2 python-docx

from google import genai
from pypdf import PdfReader
from fpdf import FPDF
import os, re, json, textwrap, getpass
from google.colab import files
print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 878.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 17.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.
Dependencies installed.


## 3. Configuration / API Setup
**Where to enter your API key:** when you run the cell below, a secure prompt appears.
Paste your Gemini API key there. It is stored only in this runtime's memory (`getpass`),
never written into the notebook file, never printed.
Get a free key: https://aistudio.google.com/app/apikey


In [3]:
API_KEY = getpass.getpass("Paste your Google Gemini API key (hidden): ").strip()
assert API_KEY, "No key entered."

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.1-flash-lite"
print("Using model:", MODEL_NAME)

def llm(prompt: str, temperature: float = 0.4) -> str:
    """Single reusable LLM call. All modules route through this."""
    interaction = client.interactions.create(
        model=MODEL_NAME,
        input=prompt,
    )
    return (interaction.output_text or "").strip()

# Smoke test
print(llm("Reply with exactly: LLM OK"))

Paste your Google Gemini API key (hidden): ··········
Using model: gemini-3.1-flash-lite
LLM OK


## 4. Prompt Engineering System

In [4]:
"""
Structured prompt template: Role / Context / Instructions / User input /
Constraints / Expected output format.
Every prompt in this project is built from this one function so behaviour stays
consistent and easy to tune.
"""

def build_prompt(role, context, instructions, user_input, constraints, output_format):
    return f"""### ROLE
{role}

### CONTEXT
{context}

### INSTRUCTIONS
{instructions}

### USER INPUT
{user_input}

### CONSTRAINTS
{constraints}

### EXPECTED OUTPUT FORMAT
{output_format}
""".strip()

PROMPTS = {}

PROMPTS["resume_generate"] = lambda p: build_prompt(
    role="You are an expert ATS-aware resume writer for students and early-career candidates.",
    context="The candidate profile below is the ONLY source of truth. Do not invent employers, dates, or degrees.",
    instructions="Write a clean, professional, single-column resume in plain text with clear section headings.",
    user_input=p,
    constraints="No fabrication. No first-person pronouns. Concise bullet points. ATS-friendly.",
    output_format="Plain text with sections: Name & Contact | Summary | Skills | Education | Projects | Experience | Achievements",
)

PROMPTS["resume_analyze"] = lambda p: build_prompt(
    role="You are a senior technical recruiter and ATS specialist.",
    context="Analyze the resume below as if screening for a real role.",
    instructions="Score the resume, list strengths, weaknesses, and concrete fixes.",
    user_input=p,
    constraints="Be specific and actionable. No generic advice.",
    output_format="Structured text: OVERALL SCORE (0-100) | STRENGTHS | WEAKNESSES | ATS ISSUES | TOP 5 FIXES",
)

PROMPTS["jd_analyze"] = lambda p: build_prompt(
    role="You are a hiring manager extracting structured requirements from a job description.",
    context="Extract only what is explicitly stated or clearly implied.",
    instructions="Identify required skills, keywords, qualifications, and role expectations.",
    user_input=p,
    constraints="Do not invent requirements that are not in the JD.",
    output_format="Structured text: ROLE TITLE | REQUIRED SKILLS | PREFERRED SKILLS | KEYWORDS | QUALIFICATIONS | RESPONSIBILITIES",
)

PROMPTS["resume_jd_match"] = lambda p: build_prompt(
    role="You are an ATS matching engine.",
    context="Compare the resume against the job description objectively.",
    instructions="Compute a match percentage, list matching and missing keywords/skills, and give targeted suggestions.",
    user_input=p,
    constraints="Match % must be justified by the keyword overlap you list.",
    output_format="Structured text: MATCH % | MATCHING KEYWORDS | MISSING KEYWORDS | MATCHING SKILLS | MISSING SKILLS | SUGGESTIONS",
)

PROMPTS["skill_gap"] = lambda p: build_prompt(
    role="You are a career skills coach.",
    context="Based on the target role and current skills, identify gaps.",
    instructions="Categorize missing skills by priority and recommend what to learn first.",
    user_input=p,
    constraints="Priority must be justified by role relevance.",
    output_format="Structured text: HIGH PRIORITY | MEDIUM PRIORITY | LOW PRIORITY | RECOMMENDED NEXT STEPS",
)

PROMPTS["career_mentor"] = lambda p: build_prompt(
    role="You are a warm, pragmatic career mentor for students.",
    context="Answer the student's career question with role-specific, actionable guidance.",
    instructions="Give direct advice, then a short action plan.",
    user_input=p,
    constraints="Be encouraging but honest. No fluff.",
    output_format="ADVICE | ACTION PLAN (3-5 steps)",
)

PROMPTS["roadmap"] = lambda p: build_prompt(
    role="You are a curriculum designer for tech careers.",
    context="Design a beginner-to-advanced roadmap for the target role.",
    instructions="Phase the roadmap, list skills per phase, and recommend portfolio projects.",
    user_input=p,
    constraints="Realistic timeline. Free/low-cost resources preferred.",
    output_format="PHASE 1 (Beginner) | PHASE 2 (Intermediate) | PHASE 3 (Advanced) | PROJECTS | MILESTONES",
)

PROMPTS["interview"] = lambda p: build_prompt(
    role="You are a technical interviewer.",
    context="Generate interview questions for the given role and level.",
    instructions="Produce HR, technical, and role-specific questions with brief ideal-answer hints.",
    user_input=p,
    constraints="Questions must be relevant to the role.",
    output_format="HR QUESTIONS | TECHNICAL QUESTIONS | ROLE-SPECIFIC QUESTIONS | PREPARATION REPORT",
)

PROMPTS["cover_letter"] = lambda p: build_prompt(
    role="You are a professional cover-letter writer.",
    context="Write a tailored cover letter using the candidate profile and job description.",
    instructions="Match tone to the company. Keep it under 350 words.",
    user_input=p,
    constraints="No fabrication. No clichés like 'I am writing to apply'.",
    output_format="Plain text cover letter with greeting, 3-4 body paragraphs, closing.",
)

PROMPTS["pdf_analyze"] = lambda p: build_prompt(
    role="You are a document analyst.",
    context="Analyze the PDF text below per the requested task.",
    instructions="Follow the task instruction exactly.",
    user_input=p,
    constraints="Only use information present in the document.",
    output_format="Clear, structured response matching the task.",
)

print("Prompt system ready. Templates:", list(PROMPTS.keys()))

Prompt system ready. Templates: ['resume_generate', 'resume_analyze', 'jd_analyze', 'resume_jd_match', 'skill_gap', 'career_mentor', 'roadmap', 'interview', 'cover_letter', 'pdf_analyze']


## 5. Resume Processing (upload / paste)

In [5]:
SAMPLE_RESUME = """Alex Kumar
alex.kumar@email.com | +91-90000-00000 | Bengaluru, India | linkedin.com/in/alexkumar

SUMMARY
Final-year B.Tech CSE student with hands-on experience in Python, data analysis,
and machine learning. Built 4 end-to-end ML projects. Seeking a Data Analyst role.

SKILLS
Python, Pandas, NumPy, SQL, Matplotlib, Scikit-learn, Git, Excel, Basic Tableau

EDUCATION
B.Tech Computer Science, XYZ Institute of Technology, 2021-2025, CGPA 8.4/10

PROJECTS
- Sales Forecasting: Built a time-series model (Prophet) on 3 years of retail data; reduced forecast error by 18%.
- Customer Segmentation: K-means on 10k customer records; identified 5 actionable segments.
- Resume Screener: NLP keyword extractor in Python; 90% precision on test set.

EXPERIENCE
Data Analyst Intern, ABC Analytics (Jun-Aug 2024)
- Automated weekly reports with Pandas, saving 6 hours/week.
- Built a dashboard tracking 12 KPIs for the ops team.

ACHIEVEMENTS
- Top 5 in college hackathon (2024). Coursera ML certificate."""

def load_resume_from_upload():
    """Upload a .txt, .pdf, or .docx resume. Returns extracted text."""
    up = files.upload()
    name = next(iter(up))
    data = up[name]
    if name.lower().endswith(".pdf"):
        import io
        reader = PdfReader(io.BytesIO(data))
        return "\n".join((p.extract_text() or "") for p in reader.pages)
    if name.lower().endswith(".docx"):
        import io
        from docx import Document
        doc = Document(io.BytesIO(data))
        return "\n".join(p.text for p in doc.paragraphs)
    return data.decode("utf-8", errors="ignore")

# TEST: sample resume loads
print(SAMPLE_RESUME[:200], "...")
print("\nResume processing ready. Use load_resume_from_upload() to upload your own.")

Alex Kumar
alex.kumar@email.com | +91-90000-00000 | Bengaluru, India | linkedin.com/in/alexkumar

SUMMARY
Final-year B.Tech CSE student with hands-on experience in Python, data analysis,
and machine l ...

Resume processing ready. Use load_resume_from_upload() to upload your own.


## 6. Resume Generator

In [6]:
def generate_resume(profile_text: str) -> str:
    return llm(PROMPTS["resume_generate"](profile_text), temperature=0.3)

# TEST
sample_profile = """Name: Priya Sharma
Email: priya@email.com | Phone: +91-98765-43210 | Pune, India
Education: B.Sc. IT, 2025, 8.1 CGPA
Skills: Python, SQL, Excel, Power BI, Communication
Interests: Data Analytics, Business Intelligence
Target Role: Junior Data Analyst
Projects: College attendance dashboard in Power BI; Python script to clean CSV sales data."""

generated_resume = generate_resume(sample_profile)
print(generated_resume)

PRIYA SHARMA
Pune, India | +91-98765-43210 | priya@email.com

SUMMARY
Aspiring Data Analyst with a strong foundation in data visualization, data cleaning, and statistical analysis. Proficient in Python, SQL, and Power BI. Eager to leverage technical skills and analytical mindset to drive data-informed decision-making in a Junior Data Analyst role.

SKILLS
* Programming: Python
* Database: SQL
* Visualization: Power BI
* Tools: Microsoft Excel
* Soft Skills: Effective Communication, Analytical Thinking

EDUCATION
B.Sc. IT | 2025
CGPA: 8.1/10.0

PROJECTS
College Attendance Dashboard | Power BI
* Developed an interactive dashboard to track and visualize student attendance trends.
* Implemented data filters and slicers to enable department-wise performance analysis.

Sales Data Cleaning Script | Python
* Authored a Python script to automate the processing and cleaning of raw CSV sales data.
* Standardized data formats, removed duplicates, and prepared datasets for subsequent reporting.

EX

## 7. Resume Analyzer (ATS-style)

In [7]:
def analyze_resume(resume_text: str) -> str:
    return llm(PROMPTS["resume_analyze"](resume_text), temperature=0.3)

# TEST
resume_analysis = analyze_resume(SAMPLE_RESUME)
print(resume_analysis)

### OVERALL SCORE: 62/100

This resume is a solid baseline for an entry-level candidate but suffers from being "under-optimized." It relies too heavily on generic descriptions, lacking the technical depth required to clear ATS filters or capture a hiring manager’s attention in a competitive data market.

---

### STRENGTHS
*   **Quantifiable Impact:** You successfully included metrics (18% reduction, 6 hours/week, 90% precision), which is rare for students. Keep this up.
*   **Conciseness:** The resume is clean, easy to read, and avoids fluff.
*   **Relevant Stack:** The technical skills listed (Python/SQL/Pandas) align perfectly with modern Data Analyst job descriptions.

---

### WEAKNESSES
*   **The "Student" Trap:** Your summary feels like a student profile rather than a professional value proposition. It needs to pivot toward the *solutions* you provide to a company.
*   **Missing Technical Breadth:** In data analytics, "Basic Tableau" is a red flag. Either frame it as "Data Visua

## 8. Job Description Analyzer

In [8]:
SAMPLE_JD = """Job Title: Junior Data Analyst
Company: FinEdge Analytics
Location: Bengaluru (Hybrid)

Requirements:
- Bachelor's degree in CS, IT, Statistics, or related field
- 0-2 years experience with Python and SQL
- Proficiency in Pandas, NumPy, and data visualization (Matplotlib/Tableau/Power BI)
- Familiarity with A/B testing and basic statistics
- Strong communication and stakeholder reporting skills
- Experience with Git and version control

Responsibilities:
- Clean, transform, and analyze large datasets
- Build dashboards and reports for business teams
- Support A/B test design and analysis
- Collaborate with product and marketing stakeholders

Nice to have: Exposure to cloud (GCP/AWS), basic ML, Excel automation."""

def analyze_jd(jd_text: str) -> str:
    return llm(PROMPTS["jd_analyze"](jd_text), temperature=0.2)

# TEST
jd_analysis = analyze_jd(SAMPLE_JD)
print(jd_analysis)

**ROLE TITLE** | Junior Data Analyst

**REQUIRED SKILLS** | Python, SQL, Pandas, NumPy, Data Visualization (Matplotlib, Tableau, or Power BI), A/B testing, Basic statistics, Communication, Stakeholder reporting, Git, Version control

**PREFERRED SKILLS** | Cloud (GCP/AWS), Basic Machine Learning (ML), Excel automation

**KEYWORDS** | Data Analysis, Large datasets, Dashboarding, Reporting, Data cleaning, Data transformation

**QUALIFICATIONS** | Bachelor's degree in CS, IT, Statistics, or related field; 0-2 years of experience

**RESPONSIBILITIES** | Clean, transform, and analyze large datasets; Build dashboards and reports for business teams; Support A/B test design and analysis; Collaborate with product and marketing stakeholders


## 9. Resume–Job Match

In [9]:
def match_resume_to_jd(resume_text: str, jd_text: str) -> str:
    payload = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{jd_text}"
    return llm(PROMPTS["resume_jd_match"](payload), temperature=0.2)

# TEST
match_result = match_resume_to_jd(SAMPLE_RESUME, SAMPLE_JD)
print(match_result)

### MATCH %: 85%

**JUSTIFICATION:** 
Alex has a very strong alignment with the core technical requirements (Python, SQL, Pandas, NumPy, Visualization). The match percentage is high because the candidate meets the educational requirements, has relevant internship experience, and possesses the necessary toolset. The gap lies in the lack of specific mention of A/B testing and business-focused communication/stakeholder management, which are highlighted in the job description.

---

### MATCHING KEYWORDS
* Python
* SQL
* Pandas
* NumPy
* Matplotlib
* Tableau
* Git
* Data Analysis
* Bachelor's degree (B.Tech CSE)
* Dashboards/Reporting
* Excel

### MISSING KEYWORDS
* A/B Testing
* Statistics (explicitly mentioned as a skill)
* Stakeholder Reporting
* Cloud (GCP/AWS - "Nice to have")
* Power BI

### MATCHING SKILLS
* Data Cleaning & Transformation
* Python Scripting
* Data Visualization
* Version Control (Git)
* Project Implementation (ML/Data)

### MISSING SKILLS
* Statistical Modeling (A/B

## 10. Skill Gap Analyzer

In [10]:
def analyze_skill_gap(current_skills: str, target_role: str) -> str:
    payload = f"TARGET ROLE: {target_role}\n\nCURRENT SKILLS: {current_skills}"
    return llm(PROMPTS["skill_gap"](payload), temperature=0.3)

# TEST
gap_result = analyze_skill_gap(
    "Python, Pandas, SQL, Matplotlib, Git",
    "Junior Data Analyst"
)
print(gap_result)

As a career skills coach, I have analyzed your current stack against the expectations for a Junior Data Analyst. You have a very strong foundation in the "core engine" of data work (Python and SQL), which puts you ahead of many candidates. 

However, you are currently missing the **"Business Value"** and **"Production"** layers of the role.

---

### HIGH PRIORITY
*   **Data Visualization Tools (Tableau or Power BI):** While Matplotlib is great for exploratory analysis, businesses almost exclusively use BI tools for reporting and executive dashboards. Being able to build an interactive, shareable dashboard is a non-negotiable requirement for most Junior Data Analyst job descriptions.
*   **Excel (Advanced):** Do not underestimate this. It is still the universal language of business. You need to be proficient in VLOOKUP/XLOOKUP, Pivot Tables, and Power Query to handle ad-hoc data requests from non-technical stakeholders.

### MEDIUM PRIORITY
*   **Statistics and Probability:** You know 

## 11. AI Career Mentor

In [11]:
def ask_career_mentor(question: str, profile_context: str = "") -> str:
    payload = f"STUDENT CONTEXT: {profile_context or 'Not provided'}\n\nQUESTION: {question}"
    return llm(PROMPTS["career_mentor"](payload), temperature=0.5)

# TEST
mentor_answer = ask_career_mentor(
    "Should I focus on Power BI or Tableau first for a data analyst role?",
    profile_context="Final-year B.Sc IT student in Pune, knows Python and SQL."
)
print(mentor_answer)

### ADVICE
In the Pune job market, **Power BI** is the clear winner for a fresh graduate. Because most Indian IT service firms and mid-sized startups operate within the Microsoft ecosystem (Azure/Office 365), Power BI integration is standard. Tableau is powerful and visually superior, but it is often reserved for niche firms or high-end enterprise clients. Since you already know Python and SQL, you have the "back-end" logic down; adding Power BI will make you immediately deployable for entry-level data analyst or junior business intelligence roles. Don’t get stuck in "tool paralysis"—pick Power BI, master the DAX language, and focus on building projects that solve real business problems.

### ACTION PLAN
1.  **Master the Fundamentals:** Spend two weeks learning Power BI Desktop—focus specifically on Power Query (data cleaning) and DAX (data modeling).
2.  **SQL Integration:** Build a project where you pull data from a SQL database into Power BI. Companies care more about how you clean 

## 12. Personalized Learning Roadmap

In [12]:
def generate_roadmap(target_role: str, current_level: str, weeks: int = 12) -> str:
    payload = (f"TARGET ROLE: {target_role}\nCURRENT LEVEL: {current_level}\n"
               f"TIMELINE: {weeks} weeks")
    return llm(PROMPTS["roadmap"](payload), temperature=0.4)

# TEST
roadmap_result = generate_roadmap("Junior Data Analyst", "Beginner (Python + SQL basics)", weeks=12)
print(roadmap_result)

This roadmap is designed to transition you from basic syntax knowledge to job-ready data analysis skills in 12 weeks.

### PHASE 1: Foundations & Data Wrangling (Weeks 1–4)
*Focus: Mastering the tools to extract and clean data.*

*   **SQL:** Aggregations (GROUP BY, HAVING), Joins (INNER, LEFT), subqueries, and common table expressions (CTEs).
*   **Python (Pandas):** Data cleaning (handling nulls, duplicates), filtering, grouping, and merging DataFrames.
*   **Excel/Google Sheets:** Pivot tables, VLOOKUP/XLOOKUP, and conditional formatting.
*   **Statistics:** Mean, median, mode, standard deviation, and correlation.

### PHASE 2: Exploratory Data Analysis & Visualization (Weeks 5–8)
*Focus: Finding stories within the data and communicating them.*

*   **Data Visualization:** Principles of design (Tufte). Tool proficiency in **Tableau** or **Power BI**.
*   **Exploratory Data Analysis (EDA):** Univariate and multivariate analysis. Identifying outliers and data trends.
*   **Advanced Py

## 13. AI Interview Studio

In [13]:
def generate_interview_questions(role: str, level: str = "entry", count: int = 5) -> str:
    payload = f"ROLE: {role}\nLEVEL: {level}\nQUESTIONS PER CATEGORY: {count}"
    return llm(PROMPTS["interview"](payload), temperature=0.5)

# TEST
interview_result = generate_interview_questions("Junior Data Analyst", "entry", 5)
print(interview_result)

### HR QUESTIONS

1. **Can you describe a time you had to explain a complex technical concept to someone non-technical?**
   * *Hint:* Look for clarity, patience, and the ability to avoid jargon.
2. **Tell me about a time you made a mistake with data or a report. How did you handle it?**
   * *Hint:* Assess accountability, problem-solving, and attention to detail.
3. **How do you stay organized when working on multiple projects with different deadlines?**
   * *Hint:* Look for time management tools (Trello, Jira) and prioritization skills.
4. **Why do you want to pursue a career as a Data Analyst specifically?**
   * *Hint:* Look for passion for data-driven storytelling and curiosity.
5. **How do you handle feedback on your analysis, especially if a stakeholder disagrees with your findings?**
   * *Hint:* Look for a collaborative mindset and objective reasoning.

### TECHNICAL QUESTIONS

1. **What is the difference between an INNER JOIN and a LEFT JOIN in SQL?**
   * *Hint:* INNER JOIN

## 14. Cover Letter Generator

In [14]:
def generate_cover_letter(profile_text: str, jd_text: str, company: str = "") -> str:
    payload = (f"COMPANY: {company or 'the company'}\n\nCANDIDATE PROFILE:\n{profile_text}\n\n"
               f"JOB DESCRIPTION:\n{jd_text}")
    return llm(PROMPTS["cover_letter"](payload), temperature=0.4)

# TEST
cover_letter = generate_cover_letter(
    "Final-year B.Sc IT student, Python + SQL + Power BI, seeking Junior Data Analyst role.",
    SAMPLE_JD,
    company="FinEdge Analytics"
)
print(cover_letter)

Dear Hiring Manager,

FinEdge Analytics’ reputation for turning complex financial data into actionable strategy is exactly the environment where I want to launch my career as a Junior Data Analyst. As a final-year B.Sc. IT student, I have built a robust technical foundation that aligns directly with your needs for data cleaning, visualization, and statistical rigor.

During my academic projects, I have utilized Python (Pandas, NumPy) to transform raw datasets into clean, reliable inputs for analysis. I am highly comfortable writing complex SQL queries to extract insights from relational databases and have developed interactive dashboards in Power BI to translate technical findings into clear, stakeholder-facing reports. My experience managing code through Git has ensured that my analytical workflows remain organized and reproducible, a practice I am eager to bring to your team.

I have a firm grasp of fundamental statistics and have applied A/B testing methodologies to optimize project

## 15. PDF → AI (summarize, Q&A, extract)

In [15]:
def pdf_to_text(uploaded_bytes: bytes) -> str:
    import io
    reader = PdfReader(io.BytesIO(uploaded_bytes))
    return "\n".join((p.extract_text() or "") for p in reader.pages)

def pdf_analyze(pdf_text: str, task: str) -> str:
    payload = f"TASK: {task}\n\nDOCUMENT TEXT:\n{pdf_text[:15000]}"
    return llm(PROMPTS["pdf_analyze"](payload), temperature=0.3)

# TEST: run the analyzer on a text blob (real PDF upload available via files.upload())
pdf_demo = pdf_analyze(SAMPLE_RESUME, "Extract skills and ATS keywords from this resume.")
print(pdf_demo)

# To analyze a real PDF:
# uploaded = files.upload()
# data = list(uploaded.values())[0]
# text = pdf_to_text(data)
# print(pdf_analyze(text, "Summarize this document in 5 bullets."))

Based on the resume provided, here are the extracted skills and ATS keywords:

### **Technical Skills**
*   **Programming Languages:** Python
*   **Data Analysis & Manipulation:** Pandas, NumPy, SQL, Excel
*   **Visualization:** Matplotlib, Tableau
*   **Machine Learning:** Scikit-learn, Time-series modeling (Prophet), K-means clustering, NLP (Natural Language Processing)
*   **Tools & Version Control:** Git

### **ATS Keywords**
*   **Job Titles:** Data Analyst, B.Tech Computer Science student
*   **Core Competencies:** Data Analysis, Machine Learning, Forecasting, Customer Segmentation, Resume Screening, Automation, KPI Tracking, Reporting
*   **Methodologies/Concepts:** Time-series analysis, K-means, NLP, Data visualization, Retail data analysis
*   **Credentials:** B.Tech Computer Science, Coursera ML Certificate


## 16. Text → PDF

In [16]:
# ============================================================
# SECTION 16 — Text → PDF (robust)
# ============================================================

def _safe(text: str) -> str:
    replacements = {
        "\u2018": "'", "\u2019": "'",
        "\u201c": '"', "\u201d": '"',
        "\u2013": "-", "\u2014": "-",
        "\u2026": "...", "\u2022": "*",
        "\u00a0": " ",
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text.encode("latin-1", errors="replace").decode("latin-1")


def _break_long_words(line: str, chunk: int = 40) -> str:
    """Split unbreakable tokens so fpdf always has a space to wrap on."""
    out = []
    for word in line.split(" "):
        if len(word) > chunk:
            word = " ".join(word[i:i+chunk] for i in range(0, len(word), chunk))
        out.append(word)
    return " ".join(out)


def text_to_pdf(text: str, filename: str = "output.pdf") -> str:
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.set_margins(left=15, top=15, right=15)
    pdf.add_page()
    pdf.set_font("Helvetica", size=11)

    safe = _safe(text)
    page_width = pdf.w - pdf.l_margin - pdf.r_margin   # usable width in mm

    for raw_line in safe.split("\n"):
        line = _break_long_words(raw_line) if raw_line.strip() else " "
        # Reset x to the left margin before EVERY cell — this is the key fix.
        pdf.set_x(pdf.l_margin)
        pdf.multi_cell(page_width, 6, line)

    pdf.output(filename)
    return filename


# ---- TEST ----
test_text = (
    "Hello from AI Career Assistant Phase 1.\n"
    "This is a test PDF with a smart quote 'like this', an em-dash - like that, "
    "and a very long unbreakable token: "
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ."
)

out = text_to_pdf(test_text, "test_text.pdf")
print("Created:", out)

import os
print("Exists:", os.path.exists(out), "| Size:", os.path.getsize(out), "bytes")

from google.colab import files
files.download(out)

Created: test_text.pdf
Exists: True | Size: 1152 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 17. AI Content → PDF

In [17]:
def ai_content_to_pdf(ai_text: str, filename: str = "ai_output.pdf") -> str:
    return text_to_pdf(ai_text, filename)

# TEST: download the AI-generated resume as a PDF
out2 = ai_content_to_pdf(generated_resume, "ai_resume.pdf")
print("Created:", out2)
files.download(out2)

Created: ai_resume.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 18. End-to-End Testing

In [18]:
def run_e2e():
    results = {}
    try:
        results["resume_generate"] = bool(generate_resume("Test profile: Python, SQL, B.Tech 2025"))
        results["resume_analyze"] = bool(analyze_resume(SAMPLE_RESUME))
        results["jd_analyze"] = bool(analyze_jd(SAMPLE_JD))
        results["resume_jd_match"] = bool(match_resume_to_jd(SAMPLE_RESUME, SAMPLE_JD))
        results["skill_gap"] = bool(analyze_skill_gap("Python, SQL", "Data Analyst"))
        results["career_mentor"] = bool(ask_career_mentor("What should I learn next?"))
        results["roadmap"] = bool(generate_roadmap("Data Analyst", "Beginner"))
        results["interview"] = bool(generate_interview_questions("Data Analyst", "entry", 3))
        results["cover_letter"] = bool(generate_cover_letter("Python dev", SAMPLE_JD))
        results["pdf_analyze"] = bool(pdf_analyze(SAMPLE_RESUME, "List skills."))
    except Exception as e:
        results["error"] = str(e)
    return results

e2e = run_e2e()
for k, v in e2e.items():
    print(f"{k:20s} -> {v}")

resume_generate      -> True
resume_analyze       -> True
jd_analyze           -> True
resume_jd_match      -> True
skill_gap            -> True
career_mentor        -> True
roadmap              -> True
interview            -> True
cover_letter         -> True
pdf_analyze          -> True


## 19. Final Project Test

In [19]:
print("=" * 60)
print("PHASE 1 FINAL REPORT")
print("=" * 60)
passed = sum(1 for k, v in e2e.items() if v is True)
total = len([k for k in e2e if k != "error"])
print(f"Modules passed: {passed}/{total}")
print("Sample outputs generated for: resume, analysis, JD, match, gap, roadmap, interview, cover letter, PDF.")
print("All modules runnable top-to-bottom. Phase 1 complete.")
print("=" * 60)

PHASE 1 FINAL REPORT
Modules passed: 10/10
Sample outputs generated for: resume, analysis, JD, match, gap, roadmap, interview, cover letter, PDF.
All modules runnable top-to-bottom. Phase 1 complete.
